# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import files
uploaded = files.upload()  # select capstone_features.csv
import pandas as pd
df = pd.read_csv('capstone_features.csv')
feats = ['imp_prev30','visible_queries','rare_share','anon_share','top_query_share','pos_volatility_60d']
print(f'{len(df):,} rows loaded')

Saving capstone_features.csv to capstone_features.csv
111,247 rows loaded


## 1. Distributions

imp_prev30 is heavily right-skewed (median 565, max 585,502 — a handful of
huge pages). pos_volatility_60d ranges 0.16–112, median ~9. top_query_share
and rare_share are bounded 0–1 as expected for shares.

In [2]:
print(df[feats].describe().round(3))

       imp_prev30  visible_queries  rare_share  anon_share  top_query_share  \
count  111247.000       102203.000  102203.000  102203.000       102203.000   
mean     2290.661           22.755       0.124       0.663            0.393   
std      6942.054           52.835       0.119       0.216            0.258   
min       100.000            1.000       0.000       0.000            0.003   
25%       231.000            4.000       0.041       0.550            0.197   
50%       565.000            9.000       0.086       0.714            0.319   
75%      1801.000           23.000       0.169       0.826            0.521   
max    585502.000         7889.000       0.891       0.996            1.000   

       pos_volatility_60d  
count          111246.000  
mean               10.645  
std                 8.192  
min                 0.163  
25%                 3.973  
50%                 8.987  
75%                15.640  
max               111.947  


## 2. Signal test #1 / #2 / #3 (verdict each)
Test #1 — pos_volatility_60d: median 6.6 (not declining) vs 10.4 (declining)
— a real, meaningful gap. Verdict: holds up.

Test #2 — top_query_share: median 0.303 vs 0.330 — a small gap in the
expected direction, but weak on its own. Verdict: weak signal alone,
useful in combination (confirmed by its 0.134 importance in the model).

Test #3 — imp_prev30: median 552 vs 571 — almost no difference between
classes. Verdict: does NOT separate classes well by itself, despite being
the most intuitive feature — this matches its moderate (not top) importance
score of 0.177 in the trained model.

In [3]:
print(df.groupby('is_declining')[feats].median().round(3))

              imp_prev30  visible_queries  rare_share  anon_share  \
is_declining                                                        
0                  552.0             11.0       0.093       0.705   
1                  571.0              9.0       0.083       0.720   

              top_query_share  pos_volatility_60d  
is_declining                                       
0                       0.303               6.604  
1                       0.330              10.354  


## 3. The flag-linked test

Correlation between pos_volatility_60d and is_declining: 0.189 — positive,
modest, consistent with it being the strongest single feature but still far
from sufficient alone (baseline AUC using it solo: 0.623).

In [4]:
print(df[['pos_volatility_60d','is_declining']].corr())

                    pos_volatility_60d  is_declining
pos_volatility_60d            1.000000      0.188845
is_declining                  0.188845      1.000000


## 4. What this means in practice

The most "obvious" feature (raw impressions) barely separates the classes;
the least obvious one (position volatility) is the strongest. This is the
practical case for combining features into a model rather than a manual
impressions-only rule — a manual rule based on impressions alone would
badly underperform.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.